1 - IMPORTAÇÃO E INSPEÇÃO INICIAL DOS DADOS

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display


# Carregamento do arquivo CSV usando csv.DictReader ';'
import csv

caminho_arquivo = "Base Varejo.csv"
dados_lista = []
with open(caminho_arquivo, mode="r", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")
    for linha in leitor:
        dados_lista.append(linha)

df = pd.DataFrame(dados_lista)


# Limpeza inicial: remove espaços nos nomes e elimina colunas extras vazias
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed|^$|^\.')]


# Exibindo o número de registros (linhas) e colunas
print(df.shape)

print("\n--- Colunas encontradas no arquivo original ---")
print(df.columns.tolist())

print("\n--- Primeiras 5 linhas do dataset ---")
display(df.head())

print("\n--- Ultimas 5 linhas do dataset ---")
display(df.tail())

print("\n--- Informações de tipos de dados (brutos) ---")
df.info()

print("\n--- Informações de tipos de dados (valores ausentes) ---")
print(df.isna().sum())

print("\n--- Linhas Duplicadas ---")
print(df.duplicated().sum())

print("\n--- Retorna um DataFrame apenas com as linhas repetidas ---")
display(df[df.duplicated()])

2 - DIAGNÓSTICOS

In [ ]:
# Diagnóstico de valores inconsistentes (#N/D, espaços, etc.)
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip()

inconsistentes = ['#N/D', 'N/A', '?', 'null', 'None', '', ' ', 'nan']
inconsistencias_totais = df.isin(inconsistentes) | df.isna()

print("\n--- Informações de valores inconsistentes/ausentes em todas as colunas ---")
print(inconsistencias_totais.sum())

3 - TRATAMENTO DE TEXTOS, NULOS E DATAS

In [ ]:
# Tratamento da coluna DATA (conversão para datetime)
df["DATA"] = pd.to_datetime(df["DATA"], format="%d/%m/%Y", errors="coerce")

# Lista completa de valores falsamente preenchidos para substituição
valores_falsos = ['#N/D', '#N/A', 'N/A', '?', 'null', 'None', '', ' ', 'nan']
df["PR_CAT"] = df["PR_CAT"].replace(valores_falsos, np.nan)

# Preenche os nulos com a string exigida pela rubrica
df["PR_CAT"] = df["PR_CAT"].fillna("Sem Categoria")

# Conversão de colunas numéricas (Sprint 2 - Integer e Float)
df["CL_FHL"] = pd.to_numeric(df["CL_FHL"], errors="coerce")
df["CL_EC"] = pd.to_numeric(df["CL_EC"], errors="coerce")

# Remoção de linhas duplicadas, se existirem
duplicatas = df.duplicated().sum()
print(f"\nLinhas duplicadas encontradas: {duplicatas}")
if duplicatas > 0:
    df = df.drop_duplicates()
    print("Duplicatas removidas com sucesso.")

# Verificação do dataset após limpeza
print("\n--- Estrutura das colunas após limpeza ---")
print(df.info())

4 - ESTATÍSTICA DESCRITIVA DA COLUNA DE NÚMERO DE FILHOS (CL_FHL)

In [ ]:
# Estatísticas Descritivas para a coluna de Número de Filhos (CL_FHL)
fhl = df['CL_FHL']
estatisticas = {
    'Contagem': fhl.count(),
    'Média': fhl.mean(),
    'Mediana': fhl.median(),
    'Moda': fhl.mode()[0],
    'Desvio Padrão': fhl.std(),
    'Mínimo': fhl.min(),
    'Quartil 25% (Q1)': fhl.quantile(0.25),
    'Quartil 50% (Q2)': fhl.quantile(0.50),
    'Quartil 75% (Q3)': fhl.quantile(0.75),
    'Máximo': fhl.max()
}

display(pd.DataFrame(estatisticas, index=['Valor']).T)

5 - AGRUPAMENTOS E ANÁLISE EXPLORATÓRIA

In [ ]:
# Agrupamento 1: Vendas por Categoria de Produto
print("\n--- 1. Total de Itens Comprados por Categoria de Produto ---")
vendas_categoria = df.groupby("PR_CAT")[["CO_ID"]].count().sort_values(by="CO_ID", ascending=False)
display(vendas_categoria)

In [ ]:
# Agrupamento 2: Perfil dos Clientes por Gênero e Segmento
print("\n--- 2. Contagem de Compras por Gênero e Segmento de Cliente ---")
agrupamento_genero_seg = df.groupby(["CL_GENERO", "CL_SEG"])["CO_ID"].count().unstack()
display(agrupamento_genero_seg)

In [ ]:
# Agrupamento 3: Volume de Vendas por Ano
df["ANO"] = df["DATA"].dt.year
print("\n--- 3. Total de Itens Vendidos por Ano ---")
vendas_ano = df.groupby("ANO")[["CO_ID"]].count()
display(vendas_ano)

In [ ]:
# Verificação de Compras Distintas

compras_distintas = df["CO_ID"].nunique()
max_clientes_por_compra = df.groupby("CO_ID")["CL_ID"].nunique().max()
max_datas_por_compra = df.groupby("CO_ID")["DATA"].nunique().max()

# Monta um resumo em tabela para exibir no display
resumo_co_id = pd.DataFrame({
    "Métrica": [
        "Total de compras distintas (CO_ID.nunique())",
        "Máximo de clientes por CO_ID",
        "Máximo de datas por CO_ID"
    ],
    "Resultado": [
        f"{compras_distintas:,}",
        max_clientes_por_compra,
        max_datas_por_compra
    ]
})

display(resumo_co_id)

6 - Exportação da base de dados tratada/limpa para um novo arquivo CSV

In [21]:

df.to_csv("df_limpo.csv", sep=";", index=False, encoding="utf-8-sig")

7 - CONCLUSÕES E INSIGHTS

7.1 - Impacto da Limpeza de Dados (Reflexão sobre ETL):
O processo de ETL foi crucial para a validade das análises. Sem o tratamento de nulos e de valores fakes na coluna PR_CAT, o ruído #N/D teria sido contabilizado erroneamente como uma categoria real de produto. Da mesma forma, a conversão de CL_FHL e CL_EC para tipos numéricos foi o que viabilizou o cálculo de métricas estatísticas essenciais como média e quantis.

7.2 - Predomínio da Categoria de Alimentos:
A categoria ALIMENTOS concentra o maior volume de vendas na rede (mais de 384 mil itens), seguida por HIGIENE e LIMPEZA.

7.3 - Comportamento por Segmento e Gênero:
O Segmento B lidera o volume de compras com distribuição praticamente idêntica entre homens (234k) e mulheres (233k). O público feminino possui maior representatividade no Segmento C (119k vs 85k).

7.4 - Evolução Temporal das Vendas:
O volume de itens vendidos apresentou crescimento contínuo de 2019 a 2021 (pico de 216 mil itens em 2021). A redução em 2022 decorre do encerramento do registro de dados em agosto.

7.5 - Perfil Familiar:
A análise estatística da coluna CL_FHL indica mediana e moda iguais a 0, demonstrando que a grande maioria dos clientes cadastrados não possui filhos.

7.6 - Limitações e Problemas Remanescentes:
A base de dados não apresenta variáveis financeiras (como preço unitário ou receita total), restringindo a análise exploratória apenas à contagem de volume físico de itens vendidos.